In [2]:
from scan import register_airway_trees
from scan import register_groupwise_deformable

In [4]:
input_folder = "/Data/MEDICAL/data"

In [ ]:
register_airway_trees(input_folder, which=5)

In [ ]:
register_groupwise_deformable(input_folder, which='all')

Loading 5 image(s) from '/Data/MEDICAL/data' ...
  ATM_001_0000.nii.gz  shape=(512, 512, 679)
  ATM_002_0000.nii.gz  shape=(512, 512, 635)
  ATM_003_0000.nii.gz  shape=(512, 512, 799)
  ATM_004_0000.nii.gz  shape=(512, 512, 679)
  ATM_005_0000.nii.gz  shape=(512, 512, 741)

Building groupwise template (3 iteration(s), SyN) ...
['3', '/tmp/tmp1pu7h8zq.mat', '/tmp/tmpf2ddq4qf/img0000/out0GenericAffine.mat', '/tmp/tmpf2ddq4qf/img0001/out0GenericAffine.mat', '/tmp/tmpf2ddq4qf/img0002/out0GenericAffine.mat', '/tmp/tmpf2ddq4qf/img0003/out0GenericAffine.mat', '/tmp/tmpf2ddq4qf/img0004/out0GenericAffine.mat']
output_transform_filename: /tmp/tmp1pu7h8zq.mat
reference_transform_filename: NULL
[0/5]: AFFINE: /tmp/tmpf2ddq4qf/img0000/out0GenericAffine.mat
[1/5]: AFFINE: /tmp/tmpf2ddq4qf/img0001/out0GenericAffine.mat
[2/5]: AFFINE: /tmp/tmpf2ddq4qf/img0002/out0GenericAffine.mat
[3/5]: AFFINE: /tmp/tmpf2ddq4qf/img0003/out0GenericAffine.mat
[4/5]: AFFINE: /tmp/tmpf2ddq4qf/img0004/out0GenericAffine.ma

In [ ]:
import nibabel as nib
import numpy as np
from skimage import measure
import trimesh
# ==========================================
# CONFIGURATION
# ==========================================
# Define your input file path here
file_path = '/Data/MEDICAL/data/registered/ATM_005_0000_R.nii.gz'

# Automatically generate the output STL path in the same folder
output_filename = file_path.replace('.nii.gz', '_aligned.stl')

# ==========================================
# PART 1: LOAD & INSPECT
# ==========================================
print(f"Loading {file_path}...\n")
img  = nib.load(file_path)
data = img.get_fdata() # This is your 3D array of voxel values (e.g., 512x512x656)

print("--- NIfTI File Inspection ---")
print(f"Data Shape (Grid Dimensions): {data.shape}")

voxel_size = img.header.get_zooms() # This gives you the physical size of each voxel in millimeters (e.g., (0.82, 0.82, 0.5))
print(f"Voxel Size (mm): {voxel_size}")

# Save Affine Matrix to RAM
affine_matrix = img.affine # This is the 4x4 matrix that encodes the position, orientation, and scaling of the data in real-world space.
print("\nAffine Matrix (Position in space):")
print(affine_matrix)

print("\n--- Data Value Analysis ---")
min_val = np.min(data)
max_val = np.max(data)
print(f"Minimum Value: {min_val}")
print(f"Maximum Value: {max_val}")

unique_values = np.unique(data)
if len(unique_values) <= 10:
    print(f"\nCONCLUSION: This is likely a **Segmentation Mask**.")
    print(f"It only contains these specific labels: {unique_values}")
else:
    print("\nCONCLUSION: This is likely a **Raw CT/MRI Scan**.")

# ==========================================
# PART 2: PROCESS & ALIGN MESH
# ==========================================
print("\n--- Mesh Generation & Alignment ---")
print("Applying Marching Cubes algorithm...")

# CRITICAL FIX: We do NOT use the 'spacing' parameter here. 
# We extract the raw voxel coordinates so the affine matrix can do all the work later.
verts, faces, normals, values = measure.marching_cubes(data, level=0.5)

print("Constructing 3D Mesh Object...")
mesh = trimesh.Trimesh(vertices=verts, faces=faces, vertex_normals=normals)

print("Applying the full Affine Matrix spatial transformation...")
# This step applies the position, rotation, and scaling saved in RAM to the mesh
mesh.apply_transform(affine_matrix)

print("Exporting Mesh...")
mesh.export(output_filename)

print(f"\nSuccess! Spatially aligned mesh saved to:\n{output_filename}")